In [1]:
from pathlib import Path
from google.cloud import bigquery

root = Path.cwd()
if not (root / "sql").is_dir():
    root = root.parent

sql_path = root / "sql" / "09_build_scoring_snapshot_v1.sql"
assert sql_path.is_file(), f"No encontré: {sql_path}"

scoring_sql = sql_path.read_text(encoding="utf-8").strip().rstrip(";")

audit_sql = f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS customer_count,
    COUNTIF(snapshot_date != DATE '2026-09-05') AS wrong_snapshot_dates,
    COUNTIF(target_version != 'order_created_v1') AS wrong_versions,
    COUNTIF(max_feature_order_date > snapshot_date) AS future_order_features,
    COUNTIF(max_feature_item_date > snapshot_date) AS future_item_features,
    COUNTIF(
        recency_days IS NULL
        OR orders_history IS NULL
        OR orders_90d IS NULL
        OR orders_365d IS NULL
        OR customer_age_days IS NULL
        OR ordered_value_365d IS NULL
    ) AS missing_model_features,
    COUNTIF(
        orders_history < 1
        OR orders_90d > orders_365d
        OR orders_365d > orders_history
        OR recency_days < 0
        OR recency_days > customer_age_days
    ) AS inconsistent_history
FROM ({scoring_sql}) AS scoring
"""

# Reutiliza client, LOCATION y MAX_BYTES del notebook anterior;
# si este notebook es nuevo, defínelos aquí.
PROJECT_ID = "customerretentionintelligence"
LOCATION = "US"
MAX_BYTES = 1_073_741_824

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

scoring_audit = client.query(
    audit_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES),
    location=LOCATION,
).to_dataframe()

display(scoring_audit)

checks = [
    "wrong_snapshot_dates",
    "wrong_versions",
    "future_order_features",
    "future_item_features",
    "missing_model_features",
    "inconsistent_history",
]

assert scoring_audit.loc[0, "row_count"] == 79669
assert scoring_audit.loc[0, "row_count"] == scoring_audit.loc[0, "customer_count"]
assert (scoring_audit.loc[0, checks] == 0).all()

print("Snapshot de puntuación validado.")

,row_count,customer_count,wrong_snapshot_dates,wrong_versions,future_order_features,future_item_features,missing_model_features,inconsistent_history
0,79669,79669,0,0,0,0,0,0


Snapshot de puntuación validado.


In [2]:
from pathlib import Path
import numpy as np
from xgboost import XGBClassifier

FEATURES = [
    "recency_days",
    "orders_history",
    "orders_90d",
    "orders_365d",
    "customer_age_days",
    "ordered_value_365d",
]

root = Path.cwd()
if not (root / "sql").is_dir():
    root = root.parent

model_path = root / "models" / "xgb_order_created_v1_6f.json"
assert model_path.is_file(), f"No encontré el modelo: {model_path}"

scoring_model = XGBClassifier()
scoring_model.load_model(str(model_path))
assert scoring_model.get_booster().feature_names == FEATURES

# Consulta completa: por ahora solo descargamos las variables a Python.
scoring_data = client.query(
    scoring_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES),
    location=LOCATION,
).to_dataframe()

assert len(scoring_data) == 79669
assert scoring_data["user_id"].is_unique

X_scoring = scoring_data[FEATURES].astype("float64")
assert X_scoring.notna().all().all()

reorder_scores = scoring_model.predict_proba(X_scoring)[:, 1]
assert np.isfinite(reorder_scores).all()
assert ((reorder_scores >= 0) & (reorder_scores <= 1)).all()

scored = scoring_data[
    ["user_id", "snapshot_date", "target_version"]
].copy()

scored["model_version"] = "xgb_order_created_v1_6f"
scored["reorder_score"] = reorder_scores
scored["inactivity_score"] = 1 - reorder_scores

print(f"Clientes puntuados: {len(scored):,}")
display(
    scored[["reorder_score", "inactivity_score"]]
    .describe(percentiles=[0.10, 0.50, 0.90, 0.99])
    .round(4)
)

Clientes puntuados: 79,669


,reorder_score,inactivity_score
count,79669.0000,79669.0000
mean,0.0650,0.9350
std,0.0402,0.0402
min,0.0046,0.6766
10%,0.0271,0.8821
50%,0.0550,0.9450
90%,0.1179,0.9729
99%,0.2075,0.9937
max,0.3234,0.9954


In [3]:
import pandas as pd

scored["risk_percentile"] = (
    scored["reorder_score"]
    .rank(ascending=False, method="average", pct=True)
    * 100
)

scored["risk_band"] = pd.cut(
    scored["risk_percentile"],
    bins=[0, 50, 80, 90, 100],
    labels=["0-50", "50-80", "80-90", "90-100"],
    include_lowest=True,
)

assert scored["risk_percentile"].between(0, 100).all()
assert scored["risk_band"].notna().all()

band_summary = (
    scored.groupby("risk_band", observed=True)
    .agg(
        customers=("user_id", "size"),
        min_reorder_score=("reorder_score", "min"),
        mean_reorder_score=("reorder_score", "mean"),
        max_reorder_score=("reorder_score", "max"),
    )
    .reset_index()
)

display(band_summary.round(4))

,risk_band,customers,min_reorder_score,mean_reorder_score,max_reorder_score
0,0-50,39816,0.0550,0.0941,0.3234
1,50-80,23918,0.0348,0.0445,0.0550
2,80-90,8058,0.0271,0.0308,0.0347
3,90-100,7877,0.0046,0.0151,0.0271


In [4]:
from datetime import date

FEATURES = [
    "recency_days",
    "orders_history",
    "orders_90d",
    "orders_365d",
    "customer_age_days",
    "ordered_value_365d",
]

publish = scored[
    [
        "user_id",
        "snapshot_date",
        "target_version",
        "model_version",
        "reorder_score",
        "risk_percentile",
        "risk_band",
    ]
].merge(
    scoring_data[["user_id", *FEATURES]],
    on="user_id",
    validate="one_to_one",
)

publish["snapshot_date"] = pd.to_datetime(publish["snapshot_date"]).dt.date
publish["risk_band"] = publish["risk_band"].astype("string")

assert len(publish) == 79_669
assert publish["user_id"].is_unique
assert publish["snapshot_date"].eq(date(2026, 9, 5)).all()
assert publish["reorder_score"].between(0, 1).all()
assert publish["risk_percentile"].between(0, 100).all()
assert publish.notna().all().all()

print(f"Filas listas para publicar: {len(publish):,}")
print(f"Fecha de corte: {publish['snapshot_date'].iloc[0]}")
print(f"Versión del modelo: {publish['model_version'].iloc[0]}")
display(publish.head())

Filas listas para publicar: 79,669
Fecha de corte: 2026-09-05
Versión del modelo: xgb_order_created_v1_6f


,user_id,snapshot_date,target_version,model_version,reorder_score,risk_percentile,risk_band,recency_days,orders_history,orders_90d,orders_365d,customer_age_days,ordered_value_365d
0,28721,2026-09-05,order_created_v1,xgb_order_created_v1_6f,0.054988,50.279281,50-80,390,1,0,0,390,0.0
1,21084,2026-09-05,order_created_v1,xgb_order_created_v1_6f,0.057179,45.660797,0-50,540,2,0,0,976,0.0
2,64936,2026-09-05,order_created_v1,xgb_order_created_v1_6f,0.009322,96.165384,90-100,417,4,0,0,825,0.0
3,91066,2026-09-05,order_created_v1,xgb_order_created_v1_6f,0.045582,63.367809,50-80,1232,1,0,0,1232,0.0
4,84441,2026-09-05,order_created_v1,xgb_order_created_v1_6f,0.049249,55.214073,50-80,507,2,0,0,1159,0.0


In [5]:
from google.api_core.exceptions import NotFound
from google.cloud import bigquery

table_id = f"{PROJECT_ID}.retention_ml.customer_scores_v1"
payload = publish.copy()
payload["prediction_timestamp"] = pd.Timestamp.now(tz="UTC")

schema = [
    bigquery.SchemaField("user_id", "INTEGER"),
    bigquery.SchemaField("snapshot_date", "DATE"),
    bigquery.SchemaField("target_version", "STRING"),
    bigquery.SchemaField("model_version", "STRING"),
    bigquery.SchemaField("reorder_score", "FLOAT"),
    bigquery.SchemaField("risk_percentile", "FLOAT"),
    bigquery.SchemaField("risk_band", "STRING"),
    bigquery.SchemaField("recency_days", "INTEGER"),
    bigquery.SchemaField("orders_history", "INTEGER"),
    bigquery.SchemaField("orders_90d", "INTEGER"),
    bigquery.SchemaField("orders_365d", "INTEGER"),
    bigquery.SchemaField("customer_age_days", "INTEGER"),
    bigquery.SchemaField("ordered_value_365d", "FLOAT"),
    bigquery.SchemaField("prediction_timestamp", "TIMESTAMP"),
]

try:
    client.get_table(table_id)
except NotFound:
    pass
else:
    raise RuntimeError(
        f"{table_id} ya existe. No se cargaron filas ni se sobrescribió nada."
    )

job_config = bigquery.LoadJobConfig(
    schema=schema,
    create_disposition=bigquery.CreateDisposition.CREATE_IF_NEEDED,
    write_disposition=bigquery.WriteDisposition.WRITE_EMPTY,
)

load_job = client.load_table_from_dataframe(
    payload,
    table_id,
    job_config=job_config,
    location=LOCATION,
)
load_job.result()

saved_table = client.get_table(table_id)
print(f"Tabla creada: {table_id}")
print(f"Filas cargadas: {saved_table.num_rows:,}")
print(f"Job ID: {load_job.job_id}")

Tabla creada: customerretentionintelligence.retention_ml.customer_scores_v1
Filas cargadas: 79,669
Job ID: 99f88865-fb97-4e6e-9c15-5a952a89acc6


In [6]:
audit_sql = f"""
SELECT
  COUNT(*) AS rows_saved,
  COUNT(DISTINCT user_id) AS unique_customers,
  COUNTIF(snapshot_date IS DISTINCT FROM DATE '2026-09-05') AS wrong_dates,
  COUNTIF(model_version IS DISTINCT FROM 'xgb_order_created_v1_6f') AS wrong_models,
  COUNTIF(target_version IS DISTINCT FROM 'order_created_v1') AS wrong_targets,
  COUNTIF(
    reorder_score IS NULL OR risk_percentile IS NULL
    OR risk_band IS NULL OR prediction_timestamp IS NULL
  ) AS missing_results,
  COUNTIF(
    reorder_score < 0 OR reorder_score > 1
    OR risk_percentile < 0 OR risk_percentile > 100
  ) AS out_of_range
FROM `{table_id}`
"""

saved_audit = client.query(
    audit_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES),
    location=LOCATION,
).to_dataframe()

display(saved_audit)

assert saved_audit.loc[0, "rows_saved"] == 79_669
assert saved_audit.loc[0, "unique_customers"] == 79_669
assert (saved_audit.loc[0, [
    "wrong_dates", "wrong_models", "wrong_targets",
    "missing_results", "out_of_range"
]] == 0).all()

print("Tabla publicada y validada.")

,rows_saved,unique_customers,wrong_dates,wrong_models,wrong_targets,missing_results,out_of_range
0,79669,79669,0,0,0,0,0


Tabla publicada y validada.
